# Deployment parity probe — Colab A100

Run this thin diagnostic notebook only for the immutable failed deployment named in its rendered parent values. It never trains and writes a separate, hash-verified probe report.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

os.environ['YOLO_AUTOINSTALL'] = 'false'
os.environ['ULTRALYTICS_SKIP_REQUIREMENTS_CHECKS'] = '1'
os.environ['MPLBACKEND'] = 'Agg'
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/pcb-defect-paired')
SOURCE_BUNDLE = DRIVE_ROOT / 'handoff' / 'pcb-defect-source.bundle'
SOURCE_BUNDLE_SHA256 = 'PASTE_FINAL_BUNDLE_SHA256'
EXPECTED_GIT_SHA = 'PASTE_FINAL_GIT_SHA'
PARENT_EXPERIMENT_GIT_SHA = 'PASTE_PARENT_EXPERIMENT_GIT_SHA'
PARENT_DEPLOYMENT_GATE_SHA256 = 'PASTE_PARENT_DEPLOYMENT_GATE_SHA256'
PARENT_ONNX_SHA256 = 'PASTE_PARENT_ONNX_SHA256'
REPO = Path('/content/pcb-defect-source')
PARENT_WORKSPACE = DRIVE_ROOT / 'workspaces' / PARENT_EXPERIMENT_GIT_SHA[:12]
PROBE_DIRECTORY = DRIVE_ROOT / 'probes' / f'{PARENT_EXPERIMENT_GIT_SHA[:12]}-to-{EXPECTED_GIT_SHA[:12]}'
PROBE_REPORT = PROBE_DIRECTORY / 'parity_probe.json'
if any(value.startswith('PASTE' + '_') for value in (SOURCE_BUNDLE_SHA256, EXPECTED_GIT_SHA, PARENT_EXPERIMENT_GIT_SHA, PARENT_DEPLOYMENT_GATE_SHA256, PARENT_ONNX_SHA256)):
    raise RuntimeError('Use the renderer-produced notebook with all immutable values populated')
print({'bundle': str(SOURCE_BUNDLE), 'parent_workspace': str(PARENT_WORKSPACE), 'probe_report': str(PROBE_REPORT)})

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

if sha256_file(SOURCE_BUNDLE) != SOURCE_BUNDLE_SHA256:
    raise RuntimeError('Source bundle SHA-256 mismatch')
if REPO.exists():
    observed = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO, check=True, capture_output=True, text=True).stdout.strip()
    if observed != EXPECTED_GIT_SHA:
        raise RuntimeError('Existing checkout has the wrong Git SHA; restart runtime')
else:
    subprocess.run(['git', 'clone', str(SOURCE_BUNDLE), str(REPO)], check=True)
    subprocess.run(['git', 'checkout', '--detach', EXPECTED_GIT_SHA], cwd=REPO, check=True)
status = subprocess.run(['git', 'status', '--porcelain'], cwd=REPO, check=True, capture_output=True, text=True).stdout
if status:
    raise RuntimeError(f'Source checkout is dirty: {status}')
print('SOURCE GATE PASS', EXPECTED_GIT_SHA)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.11.18'], check=True)
UV = shutil.which('uv')
if not UV:
    raise RuntimeError('uv installation failed')
subprocess.run([UV, 'sync', '--locked', '--no-editable', '--reinstall-package', 'pcb-defect', '--extra', 'train', '--group', 'eval'], cwd=REPO, check=True)
VENV_PYTHON = REPO / '.venv' / 'bin' / 'python'
if not VENV_PYTHON.is_file():
    raise RuntimeError(f'Locked environment Python is missing: {VENV_PYTHON}')
def runtime_contract_state(label: str) -> dict[str, object]:
    command = [
        str(VENV_PYTHON),
        '-m',
        'pcb_defect.runtime_contract',
        '--require-cuda-provider',
    ]
    result = subprocess.run(command, cwd=REPO, text=True, capture_output=True)
    print(f'[{label}] returncode={result.returncode}')
    if result.stdout:
        print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='', file=sys.stderr)
    if result.returncode:
        raise RuntimeError(f'{label} FAILED')
    lines = [line for line in result.stdout.splitlines() if line.strip()]
    if not lines:
        raise RuntimeError(f'{label} returned no runtime state')
    try:
        return json.loads(lines[-1])
    except json.JSONDecodeError as exc:
        raise RuntimeError(f'{label} returned invalid runtime JSON') from exc

LOCKED_RUNTIME_STATE = runtime_contract_state('LOCKED RUNTIME CONTRACT')
sys.path.insert(0, str(REPO / 'src'))
from pcb_defect.notebook_runtime import run_captured_command, verify_probe_result
print('LOCKED NON-EDITABLE TRAIN/EVAL ENVIRONMENT PASS')

In [ ]:
PROBE_DIRECTORY.mkdir(parents=True, exist_ok=True)
command = [str(VENV_PYTHON), '-m', 'pcb_defect.deployment_probe', '--repo', str(REPO), '--parent-workspace', str(PARENT_WORKSPACE), '--output', str(PROBE_REPORT), '--expected-parent-git-sha', PARENT_EXPERIMENT_GIT_SHA, '--expected-gate-sha256', PARENT_DEPLOYMENT_GATE_SHA256, '--expected-onnx-sha256', PARENT_ONNX_SHA256]
command_log = PROBE_DIRECTORY / 'probe_command.log'
result = run_captured_command(command, cwd=REPO, log_path=command_log, label='PARITY PROBE')
print(result.stdout, end='')
if result.stderr:
    print(result.stderr, end='', file=sys.stderr)
report = verify_probe_result(PROBE_REPORT, expected_parent_git_sha=PARENT_EXPERIMENT_GIT_SHA, expected_gate_sha256=PARENT_DEPLOYMENT_GATE_SHA256, expected_onnx_sha256=PARENT_ONNX_SHA256)
print('PARITY PROBE PASS', PROBE_REPORT, hashlib.sha256(PROBE_REPORT.read_bytes()).hexdigest())